#The Join & Weather data processing

###Prep for the join

In [0]:
# Airline Data    
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/")

# Change to one-year: 
# df_flights = df_flights.filter(df_flights["YEAR"] == 2015)

df_flights = df_flights.filter(df_flights["YEAR"] < 2020)

print(f"Row count :{df_flights.count()}")
display(df_flights)

In [0]:

# Track columns before dropping
original_columns = df_flights.columns

# Drop columns with high null count - flight data
df_flights = df_flights.drop("CANCELLATION_CODE") # >80% missing
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV_")]) # >80% missing and 1 data leakage
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV1_")]) # >80% missing
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV2_")]) # >80% missing
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV3_")]) # >80% missing
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV4_")]) # >80% missing
df_flights = df_flights.drop(*[c for c in df_flights.columns if c.startswith("DIV5_")]) # >80% missing

# Count dropped features
dropped_features = list(set(original_columns) - set(df_flights.columns))
dropped_features_count = len(dropped_features)

print(f"Number of features before dropping: {len(original_columns)}")
print(f"Number of features after dropping: {len(df_flights.columns)}")
print(f"Number of features dropped: {dropped_features_count}")
print(f"Columns dropped: {sorted(dropped_features)}")

display(df_flights.orderBy("ORIGIN","FL_DATE","CRS_DEP_TIME"))

In [0]:
df_flights_deduped = df_flights.dropDuplicates()
display(df_flights_deduped.count())

In [0]:
from pyspark.sql.functions import split, col, floor, expr, try_make_timestamp, timestamp_add

df_flights_with_dep_datetime = df_flights_deduped.withColumn(
    "dep_hour", floor(col("CRS_DEP_TIME") / 100).cast("int")
).withColumn(
    "dep_min", (col("CRS_DEP_TIME") % 100).cast("int")
).withColumn(
    "scheduled_dep_datetime_local",
    try_make_timestamp(
        col("YEAR"),
        col("MONTH"),
        col("DAY_OF_MONTH"),
        col("dep_hour"),
        col("dep_min"),
        expr("0")
    )
).withColumn(
    "scheduled_dep_datetime_local_minus_2hr",
    timestamp_add("hour", expr("-2"), col("scheduled_dep_datetime_local"))
)

display(df_flights_with_dep_datetime.select("FL_DATE", "CRS_DEP_TIME", "scheduled_dep_datetime_local", "scheduled_dep_datetime_local_minus_2hr"))

In [0]:
df_unique_origins = df_flights.select("ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_AIRPORT_ID").distinct()
display(df_unique_origins)

In [0]:
import requests

url = (
    "https://raw.githubusercontent.com/open-aviation-data/airports/main/data/airports.csv"
)
local_tmp_path = "/tmp/airports.csv"
dbfs_path = "dbfs:/student-groups/Group_3_1/airports.csv"

response = requests.get(url)
with open(local_tmp_path, "wb") as f:
    f.write(response.content)

dbutils.fs.cp(
    f"file:{local_tmp_path}",
    dbfs_path
)

df_airports_new = (
    spark.read.format("csv")
    .option("header", True)
    .load(dbfs_path)
)
display(df_airports_new)

## FUTURE: ADD IN AIRPORT TYPE

In [0]:
# Joining on iata works for all but one of our aiports. The last one joins in icao. 
df_airports_with_lat_lon = df_unique_origins.join(
    df_airports_new,
    (df_airports_new["iata"] == df_unique_origins["ORIGIN"]) | (df_airports_new["icao"] == df_unique_origins["ORIGIN"]),
    "left"
).filter(df_airports_new["country"].isin("US", "AS", "GU", "PR", "VI")) # US and territories only

display(df_airports_with_lat_lon)

In [0]:
# Weather data
from pyspark.sql.functions import to_timestamp, hour, minute, lag, col, unix_timestamp
from pyspark.sql.window import Window

df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/")

# Change to one-year: 
# df_weather = df_weather.filter(df_weather["YEAR"] == 2015)

df_weather = df_weather.filter(df_weather["YEAR"] < 2020)

df_weather = df_weather.withColumn("DATE", to_timestamp("DATE"))

print(f"Number of rows in weather data: {df_weather.count()}")
display(df_weather)

In [0]:
# Track columns before dropping
original_weather_columns = df_weather.columns

# drop columns with high null count, are not useful or are data leakage - weather data
df_weather = df_weather.drop( "AWND", "CDSD", "CLDD", "DSNW", "HDSD", "HTDD", "NormalsCoolingDegreeDay", "NormalsHeatingDegreeDay", "WindEquipmentChangeDate") # >80% missing
df_weather = df_weather.drop(*[c for c in df_weather.columns if c.startswith("Daily")]) # >80% missing
df_weather = df_weather.drop(*[c for c in df_weather.columns if c.startswith("Monthly")]) # >80% missing
df_weather = df_weather.drop(*[c for c in df_weather.columns if c.startswith("ShortDuration")]) # >80% missing
df_weather = df_weather.drop(*[c for c in df_weather.columns if c.startswith("Backup")])# >80% missing

# Count dropped features
dropped_weather_features = list(set(original_weather_columns) - set(df_weather.columns))
dropped_weather_features_count = len(dropped_weather_features)

print(f"Number of features before dropping: {len(original_weather_columns)}")
print(f"Number of features after dropping: {len(df_weather.columns)}")
print(f"Number of features dropped: {dropped_weather_features_count}")
print(f"Columns dropped: {sorted(dropped_weather_features)}")

display(df_weather)

In [0]:
df_unique_weather_station = (
    df_weather.select("STATION", "LATITUDE", "LONGITUDE", "NAME")
    .distinct()
    .na.drop(subset=["LATITUDE", "LONGITUDE"])
)

from pyspark.sql.functions import split, trim, regexp_extract, size, col, when

# Split NAME by comma, trim, and extract state/country logic
df_unique_weather_station = df_unique_weather_station.withColumn(
    "name_split", split(col("NAME"), ",")
).withColumn(
    "after_comma", trim(col("name_split").getItem(1))
).withColumn(
    "country",
    when(
        size(split(col("after_comma"), " ")) == 2,
        split(col("after_comma"), " ").getItem(1)
    ).otherwise(
        split(col("after_comma"), " ").getItem(0)
    )
).withColumn(
    "state",
    when(
        size(split(col("after_comma"), " ")) == 2,
        split(col("after_comma"), " ").getItem(0)
    )
).drop("name_split", "after_comma")

# Filter weather by country for US & territories only - some of the next closest airports are in different countries (i.e. mexico and el paso)
df_unique_weather_station = df_unique_weather_station.filter(df_unique_weather_station["country"].isin("US", "GU", "RQ", "GQ", "AS", "AQ", "MX", "CA", "VQ"))

print("rows:", df_unique_weather_station.count())
display(df_unique_weather_station)

In [0]:
from pyspark.sql.functions import radians, sin, cos, atan2, sqrt, col

# Add aliases to avoid ambiguous column names
df_ws = df_unique_weather_station.alias("ws")
df_air = df_airports_with_lat_lon.alias("air")

# Cross join
df_cross = df_air.crossJoin(df_ws)

# Haversine formula for distance in meters
R = 6371000  # Earth radius in meters
meters_to_miles = 0.000621371

df_with_distance = df_cross.withColumn(
    "distance_meters",
    R * 2 * atan2(
        sqrt(
            sin((radians(col("ws.LATITUDE")) - radians(col("air.latitude"))) / 2) ** 2 +
            cos(radians(col("ws.LATITUDE"))) * cos(radians(col("air.latitude"))) *
            sin((radians(col("ws.LONGITUDE")) - radians(col("air.longitude"))) / 2) ** 2
        ),
        sqrt(
            1 - (
                sin((radians(col("ws.LATITUDE")) - radians(col("air.latitude"))) / 2) ** 2 +
                cos(radians(col("ws.LATITUDE"))) * cos(radians(col("air.latitude"))) *
                sin((radians(col("ws.LONGITUDE")) - radians(col("air.longitude"))) / 2) ** 2
            )
        )
    )
).withColumn(
    "distance_miles",
    col("distance_meters") * meters_to_miles
)

print("rows:", df_with_distance.count())
display(df_with_distance)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, min_by, min

# select the 1st-5th closest weather stations for each airport. Only include ones that are at least 50 miles from the airport

df_flat = df_with_distance.select(
    col("ORIGIN"),
    col("air.latitude").alias("origin_airport_latitude"),
    col("air.longitude").alias("origin_airport_longitude"),
    col("timezone"),
    col("air.name").alias("airport_name"),
    col("ws.NAME"),
    col("STATION"),
    col("distance_meters"),
    col("distance_miles"), 
    col("ws.state"), 
    col("ws.country"), 
    col("ws.LATITUDE"), 
    col("ws.LONGITUDE"), 
    col("airport_type")
)

df_airport_weather_station_pairing = (
    df_flat
    .groupBy("ORIGIN", "origin_airport_latitude", "origin_airport_longitude", "timezone", "airport_name")
    .agg(
        min_by("ws.NAME", "distance_meters").alias("weather_station_name"),
        min_by("STATION", "distance_meters").alias("STATION"),
        min_by("state", "distance_meters").alias("weather_state"), 
        min_by("country", "distance_meters").alias("weather_country"),
        min_by("ws.LATITUDE", "distance_meters").alias("weather_station_latitude"),
        min_by("ws.LONGITUDE", "distance_meters").alias("weather_station_longitude"),
        min_by("airport_type", "distance_meters").alias("airport_type"),
        min("distance_meters").alias("min_distance_meters"),
        min("distance_miles").alias("min_distance_miles"), 
    )
)

# Add row_number by distance for each airport
df_flat_ranked = df_flat.withColumn(
    "distance_rank",
    row_number().over(Window.partitionBy("ORIGIN").orderBy(col("distance_meters").asc()))
)

# Get second closest station info
df_second_closest = df_flat_ranked.filter(col("distance_rank") == 2).select(
    col("ORIGIN"),
    col("ws.NAME").alias("second_weather_station_name"),
    col("STATION").alias("second_STATION"),
    col("distance_meters").alias("second_min_distance_meters"),
    col("distance_miles").alias("second_min_distance_miles"),
    col("ws.state").alias("second_weather_state"),
    col("ws.country").alias("second_weather_country"), 
    col("ws.LATITUDE").alias("second_weather_station_latitude"),
    col("ws.LONGITUDE").alias("second_weather_station_longitude")
)

# Join second closest info to main pairing
df_airport_weather_station_pairing = df_airport_weather_station_pairing.join(
    df_second_closest.filter(col("second_min_distance_meters") < 50000), # no more than 50 km away 
    on="ORIGIN",
    how="left"
)

# Get third closest station info
df_third_closest = df_flat_ranked.filter(col("distance_rank") == 3).select(
    col("ORIGIN"),
    col("ws.NAME").alias("third_weather_station_name"),
    col("STATION").alias("third_STATION"),
    col("distance_meters").alias("third_min_distance_meters"),
    col("distance_miles").alias("third_min_distance_miles"),
    col("ws.state").alias("third_weather_state"),
    col("ws.country").alias("third_weather_country"), 
    col("ws.LATITUDE").alias("third_weather_station_latitude"),
    col("ws.LONGITUDE").alias("third_weather_station_longitude")
)

# Join third closest info to main pairing
df_airport_weather_station_pairing = df_airport_weather_station_pairing.join(
    df_third_closest.filter(col("third_min_distance_meters") < 50000),
    on="ORIGIN",
    how="left"
)

# Get fourth closest station info
df_fourth_closest = df_flat_ranked.filter(col("distance_rank") == 4).select(
    col("ORIGIN"),
    col("ws.NAME").alias("fourth_weather_station_name"),
    col("STATION").alias("fourth_STATION"),
    col("distance_meters").alias("fourth_min_distance_meters"),
    col("distance_miles").alias("fourth_min_distance_miles"),
    col("ws.state").alias("fourth_weather_state"),
    col("ws.country").alias("fourth_weather_country"), 
    col("ws.LATITUDE").alias("fourth_weather_station_latitude"),
    col("ws.LONGITUDE").alias("fourth_weather_station_longitude")
)

# Join fourth closest info to main pairing
df_airport_weather_station_pairing = df_airport_weather_station_pairing.join(
    df_fourth_closest.filter(col("fourth_min_distance_meters") < 50000),
    on="ORIGIN",
    how="left"
)

# Get fifth closest station info
df_fifth_closest = df_flat_ranked.filter(col("distance_rank") == 5).select(
    col("ORIGIN"),
    col("ws.NAME").alias("fifth_weather_station_name"),
    col("STATION").alias("fifth_STATION"),
    col("distance_meters").alias("fifth_min_distance_meters"),
    col("distance_miles").alias("fifth_min_distance_miles"),
    col("ws.state").alias("fifth_weather_state"),
    col("ws.country").alias("fifth_weather_country"), 
    col("ws.LATITUDE").alias("fifth_weather_station_latitude"),
    col("ws.LONGITUDE").alias("fifth_weather_station_longitude")
)

# Join fifth closest info to main pairing
df_airport_weather_station_pairing = df_airport_weather_station_pairing.join(
    df_fifth_closest.filter(col("fifth_min_distance_meters") < 50000),
    on="ORIGIN",
    how="left"
)

display(df_airport_weather_station_pairing)

### Weather data processing

In [0]:
import plotly.graph_objects as go

row = df_airport_weather_station_pairing.filter(col("ORIGIN") == "OAK").limit(1).collect()[0]

lats = [
    row["origin_airport_latitude"],
    row["weather_station_latitude"],
    row["second_weather_station_latitude"],
    row["third_weather_station_latitude"],
    row["fourth_weather_station_latitude"],
    row["fifth_weather_station_latitude"]
]
lons = [
    row["origin_airport_longitude"],
    row["weather_station_longitude"],
    row["second_weather_station_longitude"],
    row["third_weather_station_longitude"],
    row["fourth_weather_station_longitude"],
    row["fifth_weather_station_longitude"]
]
labels = [
    "Airport",
    "1st Weather Station",
    "2nd Weather Station",
    "3rd Weather Station",
    "4th Weather Station",
    "5th Weather Station"
]

# Remove None values
plot_lats, plot_lons, plot_labels = zip(*[(float(lat), float(lon), label) for lat, lon, label in zip(lats, lons, labels) if lat is not None and lon is not None])

fig = go.Figure(go.Scattermapbox(
    lat=plot_lats,
    lon=plot_lons,
    mode='markers+text',
    marker=dict(size=[12, 8, 8, 8, 8, 8], color=['blue', 'red', 'red', 'red', 'red', 'red']),
    text=plot_labels,
    textposition="top right"
))

fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=8,
    mapbox_center={"lat": plot_lats[0], "lon": plot_lons[0]},
    margin={"r":0,"t":0,"l":0,"b":0}
)

fig.show()

In [0]:
from pyspark.sql.functions import date_trunc 

# Get distinct stations from 1st-5th clostest station 
distinct_stations = (
    df_airport_weather_station_pairing
    .select(col("STATION").alias("STATION"))
    .union(
        df_airport_weather_station_pairing.select(col("second_STATION").alias("STATION"))
    )
    .union(
        df_airport_weather_station_pairing.select(col("third_STATION").alias("STATION"))
    )
    .union(
        df_airport_weather_station_pairing.select(col("fourth_STATION").alias("STATION"))
    )
    .union(
        df_airport_weather_station_pairing.select(col("fifth_STATION").alias("STATION"))
    )
    .distinct()
    .filter(col("STATION").isNotNull())
)

print("Weather stations to process data for:", distinct_stations.count())

df_weather_filter = df_weather.filter(
    col("STATION").isin([row['STATION'] for row in distinct_stations.collect()])
)

print("rows:", df_weather_filter.count())
display(df_weather_filter) #.orderBy("STATION", "DATE")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last, col, date_trunc

df_weather_hour = df_weather_filter.withColumn(
    "hour_trunc", date_trunc("hour", col("DATE"))
)

# Downsample all columns except station 
cols = df_weather_filter.columns
if "STATION" in cols: cols.remove("STATION")

w = Window.partitionBy("STATION", "hour_trunc").orderBy(col("DATE").asc()).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Get the last value per hour
for c in cols:
    df_weather_hour = df_weather_hour.withColumn(
        f"{c}_last",
        last(col(c), ignorenulls=True).over(w)
    )

# group by station and hour_trunc to do the downsample and use the last value from above
df_weather_hourly = (
    df_weather_hour
    .groupBy("STATION", "hour_trunc")
    .agg(
        *[last(f"{c}_last", ignorenulls=True).alias(c) for c in cols]
    )
    .select("STATION", "hour_trunc", *cols)
    .drop("YEAR")
)

print("rows:", df_weather_hourly.count())
display(df_weather_hourly)#.orderBy("STATION", "DATE")

In [0]:
from pyspark.sql.functions import coalesce, min as spark_min, max as spark_max, sequence, explode, col, expr

# Filter weather to the stations for the ones in the flight data & add second closest staion 
df_weather_hourly_main_station = df_weather_hourly.join(
    df_airport_weather_station_pairing.select("STATION", "second_STATION", "third_STATION", "fourth_STATION", "fifth_STATION"),
    on="STATION",
    how="inner"
)

# Make sure there is a row for every hour for each station

# Get min and max timestamp for each station, plus station metadata
df_station_range = df_weather_hourly_main_station.groupBy("STATION", "LATITUDE", "LONGITUDE", "ELEVATION", "NAME", "second_STATION", "third_STATION", "fourth_STATION", "fifth_STATION").agg(
    spark_min("hour_trunc").alias("min_hour"),
    spark_max("hour_trunc").alias("max_hour")
)

# Generate all hourly timestamps for each station
df_station_hours = df_station_range.withColumn(
    "hourly_date",
    explode(
        sequence(
            col("min_hour"),
            col("max_hour"),
            expr("interval 1 hour")
        )
    )
)

# Join to weather data
df_station_hours = df_station_hours.alias("hours")
df_weather_hourly_main_station = df_weather_hourly_main_station.alias("weather")

df_weather_full_main_station = df_station_hours.join(
    df_weather_hourly_main_station,
    (col("hours.STATION") == col("weather.STATION")) & 
    (col("hours.hourly_date") == col("weather.hour_trunc")),
    "left"
)

cols_select = [c for c in df_weather_hourly.columns if c not in ("STATION", "hour_trunc", "LATITUDE", "LONGITUDE", "ELEVATION", "NAME", "second_STATION", "third_STATION", "fourth_STATION", "fifth_STATION")]

# Use station metadata from df_station_hours (already included from df_station_range)
df_weather_full_main_station = (
    df_weather_full_main_station
    .select(
        col("hours.STATION").alias("STATION"),
        col("hours.hourly_date").alias("hour_trunc"),
        col("hours.LATITUDE"), 
        col("hours.LONGITUDE"), 
        col("hours.ELEVATION"), 
        col("hours.NAME"),
        col("hours.second_STATION"),
        col("hours.third_STATION"),
        col("hours.fourth_STATION"),
        col("hours.fifth_STATION"),
        *cols_select
    )
)


# create copies of the weather data to join with the next nearest weather station and give them alias's that we can refer to then with in the join
df_weather_full_main_station = df_weather_full_main_station.alias("weather_main_station")
df_weather_hourly_backup = df_weather_hourly
df_weather_hourly_backup = df_weather_hourly_backup.alias("weather_backup")
df_weather_hourly_backup3 = df_weather_hourly
df_weather_hourly_backup3 = df_weather_hourly_backup3.alias("weather_backup3")
df_weather_hourly_backup4 = df_weather_hourly
df_weather_hourly_backup4 = df_weather_hourly_backup4.alias("weather_backup4")
df_weather_hourly_backup5 = df_weather_hourly
df_weather_hourly_backup5 = df_weather_hourly_backup5.alias("weather_backup5")

# List of weather columns to add backup 
backup_cols = df_weather_hourly.columns

# Join with the next closest weather station for the 1st-5th weather stations
df_weather_hourly_next_closest = df_weather_full_main_station.join(
    df_weather_hourly_backup, 
    on=[
        col("second_STATION") == col("weather_backup.STATION"),
        col("weather_main_station.hour_trunc") == col("weather_backup.hour_trunc")
    ],
    how="left"
).join(
    df_weather_hourly_backup3, 
    on=[
        col("third_STATION") == col("weather_backup3.STATION"),
        col("weather_main_station.hour_trunc") == col("weather_backup3.hour_trunc")
    ], 
    how="left"
).join(
    df_weather_hourly_backup4, 
    on=[
        col("fourth_STATION") == col("weather_backup4.STATION"),
        col("weather_main_station.hour_trunc") == col("weather_backup4.hour_trunc")
    ],
    how="left"
).join(
    df_weather_hourly_backup5, 
    on=[
        col("fifth_STATION") == col("weather_backup5.STATION"),
        col("weather_main_station.hour_trunc") == col("weather_backup5.hour_trunc")
    ],
    how="left"
).select(
    "weather_main_station.*",  # all weather columns
    *[col(f"weather_backup.{c}").alias(f"backup_{c}") for c in backup_cols],
    *[col(f"weather_backup3.{c}").alias(f"backup3_{c}") for c in backup_cols],
    *[col(f"weather_backup4.{c}").alias(f"backup4_{c}") for c in backup_cols],
    *[col(f"weather_backup5.{c}").alias(f"backup5_{c}") for c in backup_cols])


# List of weather columns to fill (excluding join keys) - don't coalesce the HourlyPressureChange
weather_cols = [c for c in df_weather_hourly.columns if c not in ("STATION", "hour_trunc", "hour_trunc_minus_1hr", "DATE", "LATITUDE", "LONGITUDE", "HourlyPressureChange")]

# Fill weather data nulls with next closest weather station using coalesce
for c in weather_cols:
    df_weather_hourly_next_closest = df_weather_hourly_next_closest.withColumn(
        c,
        coalesce(col(f"weather_main_station.{c}"), col(f"backup_{c}"), col(f"backup3_{c}"), col(f"backup4_{c}"), col(f"backup5_{c}"))
    )

# Get the list of columns that start with "backup" & drop
backup_cols_to_delete = [c for c in df_weather_hourly_next_closest.columns if c.startswith("backup")]
df_weather_hourly_next_closest = df_weather_hourly_next_closest.drop(*backup_cols_to_delete)

# Get the list of columns that start with "backup3" & drop
backup_cols_to_delete3 = [c for c in df_weather_hourly_next_closest.columns if c.startswith("backup3")]
df_weather_hourly_next_closest = df_weather_hourly_next_closest.drop(*backup_cols_to_delete3)

# Get the list of columns that start with "backup4" & drop
backup_cols_to_delete4 = [c for c in df_weather_hourly_next_closest.columns if c.startswith("backup4")]
df_weather_hourly_next_closest = df_weather_hourly_next_closest.drop(*backup_cols_to_delete4)

# Get the list of columns that start with "backup5" & drop 
backup_cols_to_delete5 = [c for c in df_weather_hourly_next_closest.columns if c.startswith("backup5")]
df_weather_hourly_next_closest = df_weather_hourly_next_closest.drop(*backup_cols_to_delete5)

df_weather_hourly_next_closest = df_weather_hourly_next_closest.drop("second_STATION", "third_STATION", "fourth_STATION", "fifth_STATION")

print("rows:", df_station_hours.count())
print("rows:", df_weather_hourly_main_station.count())
print("rows:", df_weather_full_main_station.count())
print("Time range:", df_station_range.select("min_hour", "max_hour").limit(1).collect())
print("rows:", df_weather_hourly_next_closest.count())
display(df_weather_hourly_next_closest)

In [0]:
from pyspark.sql.functions import col, date_trunc, last, when, unix_timestamp, lag
from pyspark.sql.window import Window


# Forward fill missing values for each station, ordered by hour_trunc, but only up to 6 hours
window_spec = Window.partitionBy("STATION").orderBy("hour_trunc").rowsBetween(-6, 0)

# List of columns to forward fill (excluding STATION and hour_trunc) 
    # don't forward fill HourlyPressureChange (we will fill with zeros)
    # don't forward fill Sunrise and Sunset (we will fill by local date time)
    # don't forward fill station colums that don't have nulls
cols_to_fill = [c for c in df_weather_hourly_next_closest.columns if c not in ("STATION", "HourlyPressureChange", "Sunrise", "Sunset", "LATITUDE", "LONGITUDE", "ELEVATION", "NAME")]

df_weather_fill = df_weather_hourly_next_closest

for c in cols_to_fill:
    df_weather_fill = df_weather_fill.withColumn(
        f"{c}_ffill",
        last(f"{c}", ignorenulls=True).over(window_spec)
    )

# Only fill if the last non-null value is within 6 hours, else set to null
df_weather_fill = df_weather_fill.withColumn(
    "last_nonnull_hour",
    last(
        col("hour_trunc"),
        ignorenulls=True
    ).over(window_spec)
)

# Create a column to calculate the difference between the last non-null value and the current value
df_weather_fill = df_weather_fill.withColumn(
    "hour_diff",
    (unix_timestamp(col("hour_trunc")) - unix_timestamp(col("last_nonnull_hour"))) / 3600
)

# limit fill to 6 hours
for c in cols_to_fill:
    df_weather_fill = df_weather_fill.withColumn(
        c,
        when(
            (col(f"{c}_ffill").isNotNull()) & (col("hour_diff") <= 6),
            col(f"{c}_ffill")
        ).otherwise(None)
    )


# Drop columns with _ffill suffix
ffill_cols_to_drop = [c for c in df_weather_fill.columns if c.endswith("_ffill")]
df_weather_fill = df_weather_fill.drop(*ffill_cols_to_drop)
df_weather_fill = df_weather_fill.drop(*["last_nonnull_hour", "hour_diff"])

# Create a pressure change column from HourlyStationPressure which has less nulls 
window_station = Window.partitionBy("STATION").orderBy("hour_trunc")
df_weather_hourly_filled = df_weather_fill.withColumn(
    "HourlyPressureChange_handcalc",
    col("HourlyStationPressure") - lag("HourlyStationPressure").over(window_station)
)
print("rows:", df_weather_hourly_filled.count())
display(df_weather_hourly_filled)

In [0]:
import pyspark.sql.functions as F

# "T" means a "trace" of precipitation, indicating an amount too small to be measured numerically
# Converting "T" to 0.0005
weather_df_filtered = df_weather_hourly_filled.withColumn(
    "HourlyPrecipitation",
    when(col("HourlyPrecipitation") == "T", "0.0005")
     .otherwise(col("HourlyPrecipitation"))
)

# Cast columns to the right datatype
num_cols_fix = ['origin_airport_latitude','origin_airport_longitude','weather_station_latitude','weather_station_longitude','ELEVATION','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyPressureTendency','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature', 'HourlyWindSpeed','HourlyPressureChange','HourlyWindGustSpeed']

casted_cols = []

for c in weather_df_filtered.columns:
    if c in num_cols_fix:
        casted_cols.append(col(c).cast('float').alias(c))
    elif c == "STATION":  # cast STATION to string
        casted_cols.append(col(c).cast('string').alias(c))
    else:
        casted_cols.append(col(c))

weather_df_filtered_casted = weather_df_filtered.select(casted_cols)
print(f"Number of rows: {weather_df_filtered.count()}\nNumber of columns: {len(weather_df_filtered.columns)}")
display(weather_df_filtered_casted)

In [0]:
import pyspark.sql.functions as F

# Note that we already ffill the weather columns with previous 6 hours in our custom join, here we will limit our imputation to previous 3 days
# Group by weather station and sort by "weather_datetime"
# Impute only for important weather columns
cols_as_is = ['HourlyPresentWeatherType','HourlySkyConditions','REPORT_TYPE','SOURCE','REM','HourlyWindDirection','weather_datetime']
cols_to_impute = ['HourlyPrecipitation', 'HourlyVisibility', 'HourlyStationPressure', 'HourlyWetBulbTemperature', 'HourlyDewPointTemperature', 'HourlyDryBulbTemperature', 'HourlyAltimeterSetting','HourlyRelativeHumidity','HourlyWindSpeed']
cols_to_derive = ['HourlyPressureTendency','HourlyPressureChange_handcalc']

# Impute weather columns with median

def impute_with_past_median(
    df,
    cols,
    station_col="STATION",
    time_col="weather_datetime",
    window_days=3      # 3 days in minutes (60*24*3)
):
    
    df = df.withColumn(
    "weather_datetime_min", #helper column for specify the imputation window
    F.datediff(col(time_col), F.lit("1970-01-01"))
)
    window_minutes = window_days * 60 * 24
    # rolling window: past `window_minutes` up to 1 minute before
    w = (
        Window
        .partitionBy(station_col)
        .orderBy("weather_datetime_min")
        .rangeBetween(-window_minutes, -1)
    )

    result = df
    for c in cols:
        median_col = f"{c}_median_window"

        # rolling median over the window
        result = result.withColumn(
            median_col,
            F.percentile_approx(col(c), 0.5).over(w)
        )

        # overwrite original column with imputed values
        result = result.withColumn(
            c,
            when(col(c).isNotNull(), col(c))
             .when(col(median_col).isNotNull(), col(median_col))
             .otherwise(F.lit(None))  # otherwise return null
        )

        result = result.drop(median_col)

    # drop helper time index
    result = result.drop("weather_datetime_min")

    return result

weather_df_imputed = impute_with_past_median(
    weather_df_filtered_casted,
    cols=cols_to_impute,
    station_col="STATION",
    time_col="DATE",
    window_days=3,
)

print(f"Number of rows: {weather_df_imputed.count()}\nNumber of columns left: {len(weather_df_imputed.columns)}")
display(weather_df_imputed)

In [0]:
# Save df_weather as a parquet file
weather_df_imputed.write.mode('overwrite').parquet(f"dbfs:/student-groups/Group_03_01/processed_weather_5y_v1.parquet")

### Weather time-based feature engineering

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as spark_sum, avg, min as spark_min, max as spark_max, lag

# Window specs
w_3hr = Window.partitionBy("STATION").orderBy("hour_trunc").rowsBetween(-2, 0)
w_6hr = Window.partitionBy("STATION").orderBy("hour_trunc").rowsBetween(-5, 0)
w_12hr = Window.partitionBy("STATION").orderBy("hour_trunc").rowsBetween(-11, 0)
w_24hr = Window.partitionBy("STATION").orderBy("hour_trunc").rowsBetween(-23, 0)

agg_features = [
    # (column, agg_func, window, feature_name)
    ("HourlyPrecipitation", spark_sum, w_3hr, "precip_3hr_sum"),
    ("HourlyPrecipitation", spark_sum, w_6hr, "precip_6hr_sum"),
    ("HourlyPrecipitation", spark_sum, w_12hr, "precip_12hr_sum"),
    ("HourlyPrecipitation", spark_sum, w_24hr, "precip_24hr_sum"),
    ("HourlyWindSpeed", avg, w_3hr, "wind_speed_3hr_avg"),
    ("HourlyWindSpeed", avg, w_6hr, "wind_speed_6hr_avg"),
    ("HourlyWindSpeed", avg, w_12hr, "wind_speed_12hr_avg"),
    ("HourlyWindSpeed", avg, w_24hr, "wind_speed_24hr_avg"),
    ("HourlyWindSpeed", spark_max, w_24hr, "wind_speed_24hr_max"),
    ("HourlyVisibility", avg, w_3hr, "visibility_3hr_avg"),
    ("HourlyVisibility", avg, w_6hr, "visibility_6hr_avg"),
    ("HourlyVisibility", avg, w_12hr, "visibility_12hr_avg"),
    ("HourlyDryBulbTemperature", avg, w_3hr, "dry_bulb_temp_3hr_avg"),
    ("HourlyDryBulbTemperature", avg, w_6hr, "dry_bulb_temp_6hr_avg"),
    ("HourlyDryBulbTemperature", avg, w_12hr, "dry_bulb_temp_12hr_avg"),
    ("HourlyDryBulbTemperature", avg, w_24hr, "dry_bulb_temp_24hr_avg"),
    ("HourlyDryBulbTemperature", spark_max, w_24hr, "dry_bulb_temp_24hr_max"),
    ("HourlyDryBulbTemperature", spark_min, w_24hr, "dry_bulb_temp_24hr_min"),
    ("HourlyRelativeHumidity", avg, w_3hr, "relative_humidity_3hr_avg"),
    ("HourlyRelativeHumidity", avg, w_6hr, "relative_humidity_6hr_avg"),
    ("HourlyRelativeHumidity", avg, w_12hr, "relative_humidity_12hr_avg"),
    ("HourlyPressureChange_handcalc", spark_min, w_24hr, "max_recent_pressure_drop_24hr"),
]

df_weather_with_features = weather_df_imputed

for col_name, func, window, feature_name in agg_features:
    df_weather_with_features = df_weather_with_features.withColumn(
        feature_name, func(col(col_name).cast("float")).over(window)
    )

from pyspark.sql.functions import col, lag, when

# Calculate the index (hours ago) of the max recent pressure drop in the last 24 hours
df_weather_with_features = df_weather_with_features.withColumn(
    "pressure_drop_24hr_list",
    F.collect_list(col("HourlyPressureChange_handcalc")).over(w_24hr)
).withColumn(
    "max_recent_pressure_drop_24hr_value",
    F.array_max(col("pressure_drop_24hr_list"))
).withColumn(
    "max_recent_pressure_drop_24hr_index",
    F.expr("""
        aggregate(
            pressure_drop_24hr_list,
            -1,
            (acc, x) -> acc + 1,
            acc -> array_position(pressure_drop_24hr_list, max_recent_pressure_drop_24hr_value) - 1
        )
    """)
).drop("pressure_drop_24hr_list", "max_recent_pressure_drop_24hr_value")

# Rate of change features
rate_change_features = [
    ("HourlyPrecipitation", "precip_rate_change"),
    ("HourlyWindSpeed", "wind_speed_rate_change"),
    ("HourlyVisibility", "visibility_rate_change"),
    ("HourlyDryBulbTemperature", "dry_bulb_temp_rate_change"),
    ("HourlyRelativeHumidity", "relative_humidity_rate_change"),
]

for col_name, feature_prefix in rate_change_features:
    for i in range(1, 4):
        df_weather_with_features = df_weather_with_features.withColumn(
            f"{feature_prefix}_{i}hr",
            col(col_name).cast("float") - lag(col(col_name).cast("float"), i).over(Window.partitionBy("STATION").orderBy("hour_trunc"))
        )


display(df_weather_with_features)

In [0]:
# Save df_weather as a parquet file
df_weather_with_features.write.mode('overwrite').parquet(f"dbfs:/student-groups/Group_03_01/processed_weather_engineered_features_5y_v1.parquet")

In [0]:
df_weather_with_features = spark.read.parquet(f"dbfs:/student-groups/Group_03_01/processed_weather_engineered_features_5y_v1.parquet")

display(df_weather_with_features)

In [0]:
df_flight_weather_join = spark.read.parquet(f"dbfs:/student-groups/Group_03_01/df_flight_weather_join_5y_v3.parquet")

display(df_flight_weather_join)

###The Join

In [0]:
from pyspark.sql.functions import to_utc_timestamp,from_utc_timestamp, col, coalesce

# To use the weather from the previous hour we are going to add one hour to the hour_trunc column before the join
df_weather_with_features = df_weather_with_features.withColumn(
    "hour_trunc_plus_1hr",
    expr("hour_trunc + INTERVAL 1 HOUR")
    )

# Alias DataFrames before join to avoid ambiguity
df_flights_with_dep_datetime = df_flights_with_dep_datetime.alias("flights")
df_weather_with_features = df_weather_with_features.alias("weather")
df_airport_weather_station_pairing = df_airport_weather_station_pairing.alias("pairing")


df_flight_weather_join = df_flights_with_dep_datetime.join(
    df_airport_weather_station_pairing,
    df_airport_weather_station_pairing["ORIGIN"] == df_flights_with_dep_datetime["ORIGIN"]
).withColumn(
    "scheduled_dep_datetime_minus_2hr_utc",
    to_utc_timestamp(col("scheduled_dep_datetime_local_minus_2hr"), col("timezone"))
).join(
        df_weather_with_features,
        (col("weather.STATION") == col("pairing.STATION")) &
        (date_trunc("hour", col("scheduled_dep_datetime_minus_2hr_utc")) == col("weather.hour_trunc_plus_1hr"))# Use the weather data point from the previos hour
    ).withColumn(
        "weather_datetime_local",
        from_utc_timestamp(col("weather.DATE"), col("timezone"))
    ).select(
        "flights.*",  # all flight columns
        "weather_datetime_local",
        "scheduled_dep_datetime_minus_2hr_utc",
        "origin_airport_latitude",
        "origin_airport_longitude",
        "timezone",
        "airport_name", 
        "airport_type",
        "weather_station_name",
        "weather.*"  # all weather columns
    )

# Rename columns
df_flight_weather_join = df_flight_weather_join.withColumnRenamed("LATITUDE", "weather_station_latitude").withColumnRenamed("LONGITUDE", "weather_station_longitude").withColumnRenamed("hour_trunc", "weather_hour_trunc_utc").withColumnRenamed("hour_trunc_minus_1hr", "weather_hour_trunc_minus_1hr").withColumnRenamed("DATE", "weather_datetime")

# Same as weather_station_name
df_flight_weather_join = df_flight_weather_join.drop("NAME", "hour_trunc_plus_1hr")

print("rows:", df_flight_weather_join.count())
display(df_flight_weather_join)#.orderBy("STATION", "weather_hour_trunc", "dep_min")

In [0]:
from pyspark.sql.functions import to_utc_timestamp, from_utc_timestamp, col, coalesce, expr

if "hour_trunc_plus_1hr" not in df_weather_with_features.columns:
    df_weather_with_features = df_weather_with_features.withColumn(
        "hour_trunc_plus_1hr",
        expr("hour_trunc + INTERVAL 1 HOUR")
    )

# Alias DataFrames before join to avoid ambiguity
df_flight_weather_join = df_flight_weather_join.alias("flight_join")
df_weather_with_features_dest = df_weather_with_features
df_weather_with_features_dest = df_weather_with_features_dest.alias("weather_dest")
df_airport_weather_station_pairing_dest = df_airport_weather_station_pairing
df_airport_weather_station_pairing_dest = df_airport_weather_station_pairing_dest.alias("pairing_dest")

# Join on DEST instead of ORIGIN
df_flight_weather_dest_join = df_flight_weather_join.join(
    df_airport_weather_station_pairing_dest,
    df_airport_weather_station_pairing_dest["ORIGIN"] == df_flight_weather_join["DEST"]
).join(
    df_weather_with_features_dest,
    (col("weather_dest.STATION") == col("pairing_dest.STATION")) &
    (expr("date_trunc('hour', scheduled_dep_datetime_minus_2hr_utc)") == col("weather_dest.hour_trunc_plus_1hr"))
).withColumn(
    "dest_weather_datetime_local",
    from_utc_timestamp(col("weather_dest.DATE"), col("pairing_dest.timezone"))
)

# Select all flight columns, and add dest_ prefix to all destination weather/airport columns
flight_cols = [c for c in df_flight_weather_join.columns]

# Only keep specific columns from pairing table for destination
pairing_keep = [
    "origin_airport_latitude",
    "origin_airport_longitude",
    "timezone",
    "airport_name",
    "airport_type",
    "weather_station_name"
]
pairing_cols = [c for c in pairing_keep]

weather_cols = [c for c in df_weather_with_features.columns if c not in ("STATION", "hour_trunc", "hour_trunc_minus_1hr", "DATE", "hour_trunc_plus_1hr", "NAME")]

# Rename columns with dest_ prefix
select_exprs = [col(f"flight_join.{c}") for c in flight_cols]
select_exprs += [col("dest_weather_datetime_local").alias("dest_weather_datetime_local")]
select_exprs += [col("scheduled_dep_datetime_minus_2hr_utc").alias("dest_scheduled_dep_datetime_minus_2hr_utc")]

for c in pairing_cols:
    select_exprs.append(col(f"pairing_dest.{c}").alias(f"dest_{c}"))
for c in weather_cols:
    select_exprs.append(col(f"weather_dest.{c}").alias(f"dest_{c}"))

df_flight_weather_dest_join = df_flight_weather_dest_join.select(*select_exprs)

df_flight_weather_dest_join = df_flight_weather_dest_join.withColumnRenamed("dest_origin_airport_latitude", "dest_airport_latitude").withColumnRenamed("dest_origin_airport_longitude", "dest_airport_longitude").withColumnRenamed("dest_LATITUDE", "dest_weather_station_latitude").withColumnRenamed("dest_LONGITUDE", "dest_weather_station_longitude")

df_flight_weather_dest_join = df_flight_weather_dest_join.drop("dest_scheduled_dep_datetime_minus_2hr_utc")

print("rows:", df_flight_weather_dest_join.count())
display(df_flight_weather_dest_join)

In [0]:
# Save df_weather as a parquet file
df_flight_weather_dest_join.write.mode('overwrite').parquet(f"dbfs:/student-groups/Group_03_01/df_flight_weather_join_5y_v4.parquet")

In [0]:
df_flight_weather_dest_join2 = spark.read.parquet(f"dbfs:/student-groups/Group_03_01/df_flight_weather_join_5y_v4.parquet")

display(df_flight_weather_dest_join2)